In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import root_mean_squared_error as rmse
import time
import tkinter as tk
from tkinter import messagebox
import math
import matplotlib as mpl
from scipy.spatial import ConvexHull
import os
# our functions
import predict_Beta_I
import choice_start_day
import plot_hyb

import warnings
warnings.filterwarnings(action='ignore')

# to account for updates when files change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## !

In [3]:
def apply_methods(seed_dirs='initial_data/initial_data_ba_10000/',
                  seed_numbers=[], on_incidence=False, 
                  switch_on_incidence=False,
                 idx_s=0, idx_e=11, show_fig_flag=False,
                 is_filename=False, sigma=0.1, gamma=0.08, perc_switch=0.01,
                  stoch=0,
                 suff_m='sw100k', suff='sw100k'):
    
    #df_seeds = pd.read_csv(df_seeds)
    if is_filename:
        col = 'file'
    else:
        col='seed_number'
        
    #seed_numbers = df_seeds[df_seeds.set=='test'][col].values[:n_seeds]

    types_start_day = ['fraq_people']#, 'roll_var', 'roll_var_seq']

    methods = ['last value','expanding mean last value',
               'median beta', 
               #'regression (day)',
               'regression beta', #'arimax',
               'lstm',
               'expdecay'
              ]

    new_labels = ['last_value', 'expanding_mean_last_value', 

            'median_beta', #'regression_day',
            'regression_beta', #'arimax',
                  'lstm_day_E_previous_I',
                  'expdecay'
                 ]

    for type_start_day in types_start_day:

        for beta_pred,new_label in zip(methods[idx_s:idx_e], 
                                       new_labels[idx_s:idx_e]):

            if 'median' in beta_pred:
                model_path = f'{suff_m}_median_beta.csv'
            elif 'regression beta' in  beta_pred:
                model_path = f'{suff_m}_regression_bt.joblib'
            elif 'lstm' in beta_pred:
                model_path = f'{suff_m}_lstm_4_001_s10'   
            else:
                model_path=''
            print('path: ', model_path)
            if stoch > 0:
                stochastic = True
            try:
                all_rmse_I, all_rmse_Inc, all_rmse_Beta, \
                all_r2, all_r2_Inc, all_r2_full, all_r2_Inc_full,\
                all_peak, \
                    execution_time, start_days = plot_hyb.main_f(I_prediction_method='seir', 
                                        count_stoch_line=stoch, 
                                        beta_prediction_method=beta_pred, 
                                        type_start_day=type_start_day, 
                                        seed_numbers=seed_numbers, 
                                        show_fig_flag=show_fig_flag,
                                        seed_dirs=seed_dirs, 
                                        sigma=sigma, gamma=gamma, 
                                        ax=None, model_path=model_path,
                                        perc_switch=perc_switch,
                                        is_filename=is_filename,
                                        on_incidence=on_incidence,
                                        switch_on_incidence=switch_on_incidence)
                
                # creating a dataframe for peaks
                all_peak = pd.DataFrame(all_peak, 
                                columns=['actual_peak_I', 'predicted_peak_I', 
                                        'actual_peak_Inc', 'predicted_peak_inc',
                                        'actual_peak_day', 'predicted_peak_day',
                                        'actual_peak_day_Inc', 'predicted_peak_day_inc'])
                # creating a dataframe for peaks RMSE, predicted time, start day
                rmse_df = pd.DataFrame({
                    'rmse_I': all_rmse_I,
                    'rmse_Inc': all_rmse_Inc,
                    'rmse_Beta': all_rmse_Beta,
                    'r2': all_r2,
                    'r2_Inc': all_r2_Inc,
                    'r2_full': all_r2_full,
                    'r2_Inc_full': all_r2_Inc_full,
                    'time_predict': execution_time,
                    f'{type_start_day}': start_days})

                # merging dataframes
                results = pd.concat([rmse_df, all_peak], axis=1)
                folder_name = seed_dirs.split('/')[-2]
                
            except FileNotFoundError:
                pass
                
            if not show_fig_flag:
                path = f'results/{folder_name}/{type_start_day}/'
                if not os.path.exists(path):
                    os.makedirs(path)
                results.to_csv(f'{path}/{new_label}_results_{suff}.csv', 
                           index=False)
                

In [4]:
seed_dirs="../new_ba_100000/"
sw = pd.read_csv('test_files.csv').values

## exps

In [ ]:

seed_dirs+sw[100][0]

In [ ]:
sw = pd.read_csv('train_files.csv').values
sw1 = pd.read_csv('test_files.csv').values
sw2 = pd.read_csv('val_files.csv').values

In [ ]:
sw.shape, sw1.shape, sw2.shape

In [ ]:
sw[100][0][:-5]

In [ ]:
N = 1e5
seed_df = pd.read_csv(seed_dirs+sw[-1][0])

temp = seed_df[['E','S']].shift([0,1])
# Inc_t = (E_t-1 - E_t) - (S_t - S_t-1)
seed_df['incidence'] = (temp['E_1'] - temp['E_0']) - \
                    (temp['S_0'] - temp['S_1'])
            
switch = seed_df[seed_df.I > 0.05*N].index[0]

plt.subplots(figsize=(6,3))
plt.plot(seed_df.incidence)
plt.plot(seed_df.I)
plt.axvline(switch, ls=':', color='red')
plt.axvline(4, ls=':', color='green')
plt.grid()
plt.xlim(0,60)
print(switch)

In [ ]:
N = 1e5
seed_df = pd.read_csv(seed_dirs+sw[-100][0])

temp = seed_df[['E','S']].shift([0,1])
# Inc_t = (E_t-1 - E_t) - (S_t - S_t-1)
seed_df['incidence'] = (temp['E_1'] - temp['E_0']) - \
                    (temp['S_0'] - temp['S_1'])
            
switch = seed_df[seed_df.I > 0.05*N].index[0]

plt.subplots(figsize=(6,3))
plt.plot(seed_df.incidence)
plt.plot(seed_df.I)
plt.axvline(switch, ls=':', color='red')
plt.axvline(4, ls=':', color='green')
plt.grid()
plt.xlim(0,60)
print(switch)

In [ ]:
N = 1e5
seed_df = pd.read_csv(seed_dirs+sw[-500][0])

temp = seed_df[['E','S']].shift([0,1])
# Inc_t = (E_t-1 - E_t) - (S_t - S_t-1)
seed_df['incidence'] = (temp['E_1'] - temp['E_0']) - \
                    (temp['S_0'] - temp['S_1'])
            
switch = seed_df[seed_df.I > 0.05*N].index[0]
plt.subplots(figsize=(6,3))
plt.plot(seed_df.incidence)
plt.plot(seed_df.I)
plt.axvline(switch, ls=':', color='red')
#plt.axvline(4, ls=':', color='green')
plt.grid()
plt.xlim(0,60)
print(switch)

In [ ]:
sw[np.mod(np.arange(sw.size),10)!=0].reshape(-1,9)[-1]

In [ ]:
sw[::10][-1]

In [ ]:
all_df2 = []
plt.subplots(figsize=(5,3))
for i in sw[np.mod(np.arange(sw.size),10)!=0].reshape(-1,9)[-1].ravel():
    d = pd.read_csv(seed_dirs+i).iloc[:60,:]
    all_df2.append(d)
    plt.plot(d.Beta, color='gray', alpha=.4)
plt.ylim(0,3e-5)

d = pd.read_csv(seed_dirs+sw[::10][-1][0]).iloc[:60,:]
all_df2.append(d)
plt.plot(d.Beta)

In [ ]:
def decay(t, b0, q, phi):
    #return [b0*np.exp(-q*tt) for tt in t]
    return [b0*((1-phi)*np.exp(-q*tt)+phi) for tt in t]


def combinedFunction(tdata, b0, q, phi):
    # single data reference passed in, extract separate data
    res = []
    
    for i in range(9):
        #print(b0[i])
        result = decay(tdata, b0[i], q, phi)
        res.append(result)
    return np.array(res).ravel()

def get_i_inc(switch, predicted_beta, gamma, delta):
    y = seed_df.iloc[switch,:4]
    S,E,I,R = seir_discrete.seir_model(y, np.arange(switch, 60), 
                            np.array(predicted_beta), gamma, delta, 
                            'det', True).T

    predicted_Inc = np.zeros((1, 60-switch))

    from_last = seed_df['incidence'].iloc[switch]
    predicted_Inc[0] = np.append(from_last, 
                               (E[:-1] - E[1:]) - \
                                   (S[1:] - S[:-1]))
    predicted_Inc[0][predicted_Inc[0]<0] = 0
    
    return I, predicted_Inc[0]

In [ ]:
all_df2[-1]

In [ ]:
seed_df[seed_df.I > 0.01*N]

In [ ]:
from scipy.optimize import curve_fit
import seir_discrete


In [ ]:
# exp. decay is good, but how to estimate q? (and phi?)
N = 1e5

seed_df = all_df2[-1]
chosen_col='incidence'
temp = seed_df[['E','S']].shift([0,1])
# Inc_t = (E_t-1 - E_t) - (S_t - S_t-1)
seed_df['incidence'] = (temp['E_1'] - temp['E_0']) - \
                    (temp['S_0'] - temp['S_1'])
            
switch = seed_df[seed_df.I > 0.01*N].index[0]
gamma = .3
delta = .2
b0 = seed_df.Beta.iloc[switch]

n = 9
#predicted_beta = [seed_df.iloc[switch]['Beta'] for i in np.arange(switch, 60)]
tdata = np.concatenate([np.arange(switch,60)-switch for i in range(n)])
comboData = np.array([all_df2[i].Beta.iloc[switch:].values for i in range(n)])
# curve fit the combined data to the combined function
coeffs, _ = curve_fit(lambda t, q, phi:combinedFunction(np.arange(switch,60)-switch, 
                                                        comboData[:,0], 
                                                        q, phi), 
                      tdata, comboData.ravel())
print('coeffs for q and phi: ',coeffs)

fig, ax = plt.subplots(1,1, figsize=(5,3))
ax.plot(comboData.T, lw=1, marker='.', color='gray', 
        alpha=.5, label =['train curves']+[None] * (8))
ax.plot(decay(tdata[:60-switch], all_df2[-1].Beta[switch], *coeffs), 
         ls='--', label='found fit for 10 curves')
ax.plot(all_df2[-1].Beta[switch:].values, color='tab:blue', 
        label='unseen Beta')
ax.grid()
ax.legend()


#phi = seed_df.iloc[switch]['Beta'] / delta#delta / seed_df.iloc[switch]['Beta']
predicted_beta = decay( np.arange(switch,60)-switch, b0, *coeffs)


I, predicted_Inc = get_i_inc(switch, predicted_beta, gamma, delta)

real_beta = seed_df.iloc[switch:]['Beta']
Ir, predicted_Incr = get_i_inc(switch, real_beta, gamma, delta)

print('switch on day ', switch)

#________________________________________________________

fig, ax = plt.subplots(1,2, figsize=(10,3))
ax = ax.flatten()

ax[0].plot(seed_df.incidence, label='Inc network',color='gray',alpha=.7)
ax[0].plot(np.arange(switch, 60), predicted_Inc, 
           ls='--', color='gray', label='Inc SEIR (from pred)')
ax[0].plot(np.arange(switch, 60), predicted_Incr, 
           ls=':', color='blue', label='Inc SEIR (from Real)')

ax[1].plot(seed_df.I, label='I network',color='black',alpha=.7)
ax[1].plot(np.arange(switch, 60), I, 
           ls='--', color='black', label='I SEIR (from pred)')
ax[1].plot(np.arange(switch, 60), Ir, 
           ls=':', color='blue', label='I SEIR (from Real)')

for ax in [ax[0], ax[1]]:
    axb = ax.twinx()
    axb.plot(seed_df.Beta, color='green')
    plt.xlim(-5,60)
    ax.grid()
    ax.axvline(switch)
    axb.plot(np.arange(switch, 60), np.array(predicted_beta), 
             ls='--', color='red')
    axb.set_ylim(0, 1e-4)
    ax.legend()
    #ax.set_zorder(100)
    
#axb.set_ylim(0, 1e-6)
plt.legend()
plt.tight_layout()

In [ ]:
# exp. decay is good, but how to estimate q? (and phi?)
N = 1e5

seed_df = all_df2[-1]
chosen_col='incidence'
temp = seed_df[['E','S']].shift([0,1])
# Inc_t = (E_t-1 - E_t) - (S_t - S_t-1)
seed_df['incidence'] = (temp['E_1'] - temp['E_0']) - \
                    (temp['S_0'] - temp['S_1'])
            
switch = seed_df[seed_df.I > 0.05*N].index[0]
gamma = .3
delta = .2
b0 = seed_df.Beta.iloc[switch]

n = 9
#predicted_beta = [seed_df.iloc[switch]['Beta'] for i in np.arange(switch, 60)]
tdata = np.concatenate([np.arange(switch,60)-switch for i in range(n)])
comboData = np.array([all_df2[i].Beta.iloc[switch:].values for i in range(n)])
# curve fit the combined data to the combined function
coeffs, _ = curve_fit(lambda t, q, phi:combinedFunction(np.arange(switch,60)-switch, 
                                                        comboData[:,0], 
                                                        q, phi), 
                      tdata, comboData.ravel())
print('coeffs for q and phi: ',coeffs)

fig, ax = plt.subplots(1,1, figsize=(5,3))
ax.plot(comboData.T, lw=1, marker='.', color='gray', 
        alpha=.5, label =['train curves']+[None] * (8))
ax.plot(decay(tdata[:60-switch], all_df2[-1].Beta[switch], *coeffs), 
         ls='--', label='found fit for 10 curves')
ax.plot(all_df2[-1].Beta[switch:].values, color='tab:blue', 
        label='unseen Beta')
ax.grid()
ax.legend()


#phi = seed_df.iloc[switch]['Beta'] / delta#delta / seed_df.iloc[switch]['Beta']
predicted_beta = decay( np.arange(switch,60)-switch, b0, *coeffs)


I, predicted_Inc = get_i_inc(switch, predicted_beta, gamma, delta)

real_beta = seed_df.iloc[switch:]['Beta']
Ir, predicted_Incr = get_i_inc(switch, real_beta, gamma, delta)

print('switch on day ', switch)

#________________________________________________________

fig, ax = plt.subplots(1,2, figsize=(10,3))
ax = ax.flatten()

ax[0].plot(seed_df.incidence, label='Inc network',color='gray',alpha=.7)
ax[0].plot(np.arange(switch, 60), predicted_Inc, 
           ls='--', color='gray', label='Inc SEIR (from pred)')
ax[0].plot(np.arange(switch, 60), predicted_Incr, 
           ls=':', color='blue', label='Inc SEIR (from Real)')

ax[1].plot(seed_df.I, label='I network',color='black',alpha=.7)
ax[1].plot(np.arange(switch, 60), I, 
           ls='--', color='black', label='I SEIR (from pred)')
ax[1].plot(np.arange(switch, 60), Ir, 
           ls=':', color='blue', label='I SEIR (from Real)')

for ax in [ax[0], ax[1]]:
    axb = ax.twinx()
    axb.plot(seed_df.Beta, color='green')
    plt.xlim(-5,60)
    ax.grid()
    ax.axvline(switch)
    axb.plot(np.arange(switch, 60), np.array(predicted_beta), 
             ls='--', color='red')
    axb.set_ylim(0, 1e-4)
    ax.legend()
    #ax.set_zorder(100)
    
#axb.set_ylim(0, 1e-6)
plt.legend()
plt.tight_layout()

In [ ]:
sw[-1][0]

In [ ]:
d = pd.read_csv(seed_dirs+sw[-1][0]).iloc[:100,:]
d

In [ ]:
d 

## back to applying

In [ ]:
#for p,l in zip([#0.01,
                0.005,
                0.0075,
                0.0125,
                0.015
               ], 
               [#'1',
                '05',
                   '075',
                '125',
                   '15'
               ]):
    print()
    print(l)

    apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=5, idx_e=6, show_fig_flag=False,
              is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,stoch=0,
              suff_m='ba100k', suff=f'ba100k_sI_fullR_{l}')

In [ ]:
%%time

#for p,l in zip([0.01,
                0.005,
                0.0075,
                0.0125,
                0.015
               ], 
               ['1',
                '05',
                   '075',
                '125',
                   '15'
               ]):
    print()
    print(l)

    apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=5, idx_e=6, show_fig_flag=False,
              is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,stoch=0,
              suff_m='sw100k', suff=f'sw100k_sI_fullR_{l}')

In [5]:
%%time

for p,l in zip([#0.03,
                #0.04,
                #0.05,
                0.06,
                0.07
               ], 
               [#'3',
                #   '4',
                #'5',
                   '6',
                '7'
               ]):
    print()
    print(l)

    apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=5, show_fig_flag=False,
              is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=p,stoch=0,
              suff_m='ba100k', suff=f'ba100k_sI_fullR_{l}')


6
path:  
last value
path:  
expanding mean last value
path:  ba100k_median_beta.csv
median beta
path:  ba100k_regression_bt.joblib
regression beta
path:  ba100k_lstm_4_001_s10
lstm

7
path:  
last value
path:  
expanding mean last value
path:  ba100k_median_beta.csv
median beta
path:  ba100k_regression_bt.joblib
regression beta
path:  ba100k_lstm_4_001_s10
lstm
CPU times: total: 52min 4s
Wall time: 46min 36s


In [ ]:
 apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[:2:1], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=5, show_fig_flag=False,
              is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=p,
              suff_m='sw100k', suff=f'hh')

In [ ]:
for p,l in zip([#0.01,
                #0.005,
                0.0075,
                0.0125,
                0.015
               ], 
               [#'1',
                #'05',
                   '075',
                '125',
                   '15'
               ]):
    print()
    print(l)
    apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=6, show_fig_flag=False,
              is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=p,
              suff_m='sw100k', suff=f'sw100k_sI_fullR_{l}')
    plt.close()

In [ ]:
p, l = 0.01, 1
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10][102:104], on_incidence=True,
              switch_on_incidence=False,
              idx_s=2, idx_e=3, show_fig_flag=False,
              is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=p, stoch=0,
              suff_m='ba100k', suff=f'ba100k_sI_fullR_{l}')
plt.close()

In [ ]:
%%time
# for lstm CPU times: total: 3h 41min 11s
# Wall time: 3h 1min 54s
#apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=4, idx_e=5, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,
             suff_m='ba100k', suff='ba100k_sI')

In [ ]:
apply_methods(seed_dirs="../hybrid_surrogate/sim_data/new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.005,
             suff_m='ba100k', suff=f'ba100k_sI_05')

In [ ]:
apply_methods(seed_dirs="../hybrid_surrogate/sim_data/new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.0075,
             suff_m='ba100k', suff=f'ba100k_sI_075')

In [ ]:
apply_methods(seed_dirs="../hybrid_surrogate/sim_data/new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.0125,
             suff_m='ba100k', suff=f'ba100k_sI_125')

In [ ]:
apply_methods(seed_dirs="../hybrid_surrogate/sim_data/new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.015,
             suff_m='ba100k', suff=f'ba100k_sI_15')

In [ ]:
#for p,l in zip([0.005,0.0075,0.0125,0.015], 
               ['05','075','1','125','15']):
    apply_methods(seed_dirs="../hybrid_surrogate/sim_data/new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              switch_on_incidence=False,
              idx_s=4, idx_e=5, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=p,
             suff_m='ba100k_', suff=f'ba100k_sI_all_{l}')
    

In [ ]:
%%time
k = 2380
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[-2:], on_incidence=True,
              switch_on_incidence=False,
              idx_s=3, idx_e=5, show_fig_flag=True,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,
             suff_m='ba100k', suff='ba100k_10')

In [ ]:
sw[::10][140:]

In [ ]:
%%time
k = 145
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10][k:k+1], on_incidence=True,
              switch_on_incidence=False,
              idx_s=3, idx_e=5, show_fig_flag=True,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01, stoch=100,
             suff_m='ba100k', suff='ba100k_10')

In [ ]:
%%time
k = 2220
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10][k:k+1], on_incidence=True,
              switch_on_incidence=False,
              idx_s=3, idx_e=5, show_fig_flag=True,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01, stoch=100,
             suff_m='ba100k', suff='ba100k_10')

In [ ]:
types_start_day = ['fraq_people']#, 'roll_var', 'roll_var_seq']

methods = ['last value','expanding mean last value',
               'median beta', 
               #'regression (day)',
               'regression beta', #'arimax',
               'lstm'
              ]

new_labels = ['last_value', 'expanding_mean_last_value', 

            'median_beta', #'regression_day',
            'regression_beta', #'arimax',
                  'lstm_day_E_previous_I'
                 ]
seed_dirs="../new_ba_100000/"

model_pathr = f'ba100k_regression_bt.joblib'
model_pathl = f'ba100k_lstm_4_001_s10'   

sigma=0.3
gamma=0.2
perc_switch=0.01
stoch=100

In [ ]:
k = 145
fig, ax = plt.subplots(2,2, figsize=(10, 6))
ax = ax.flatten()

plot_hyb.main_f(I_prediction_method='seir', count_stoch_line=100, 
                beta_prediction_method=methods[-2], 
                type_start_day=types_start_day[0], 
                seed_numbers=sw[::10][k:k+1], show_fig_flag=True,
                seed_dirs=seed_dirs, sigma=sigma, gamma=gamma, 
                ax=[ax[0]], model_path=model_pathr,perc_switch=perc_switch,
                is_filename=True,on_incidence=True,switch_on_incidence=False)
ax[0].text(-0.1, 1.1, 'a)',
           transform=ax[0].transAxes, size=15)

plot_hyb.main_f(I_prediction_method='seir', count_stoch_line=100, 
                beta_prediction_method=methods[-1], 
                type_start_day=types_start_day[0], 
                seed_numbers=sw[::10][k:k+1], show_fig_flag=True,
                seed_dirs=seed_dirs, sigma=sigma, gamma=gamma, 
                ax=[ax[1]], model_path=model_pathl,perc_switch=perc_switch,
                is_filename=True,on_incidence=True,switch_on_incidence=False);
ax[1].text(-0.1, 1.1, 'b)',
           transform=ax[1].transAxes, size=15)
k = 2320
plot_hyb.main_f(I_prediction_method='seir', count_stoch_line=100, 
                beta_prediction_method=methods[-2], 
                type_start_day=types_start_day[0], 
                seed_numbers=sw[::10][k:k+1], show_fig_flag=True,
                seed_dirs=seed_dirs, sigma=sigma, gamma=gamma, 
                ax=[ax[2]], model_path=model_pathr,perc_switch=perc_switch,
                is_filename=True,on_incidence=True,switch_on_incidence=False)
ax[2].text(-0.1, 1.1, 'c)',
           transform=ax[2].transAxes, size=15)

plot_hyb.main_f(I_prediction_method='seir', count_stoch_line=100, 
                beta_prediction_method=methods[-1], 
                type_start_day=types_start_day[0], 
                seed_numbers=sw[::10][k:k+1], show_fig_flag=True,
                seed_dirs=seed_dirs, sigma=sigma, gamma=gamma, 
                ax=[ax[3]], model_path=model_pathl,perc_switch=perc_switch,
                is_filename=True,on_incidence=True,switch_on_incidence=False);
ax[3].text(-0.1, 1.1, 'd)',
           transform=ax[3].transAxes, size=15)

plt.tight_layout()
plt.savefig(f'results/regr_lstm.pdf', format='pdf', bbox_inches='tight')

In [ ]:
ax.flatten()

In [ ]:
import joblib

In [ ]:
k = 20380

seed_dirs="../hybrid_surrogate/sim_data/new_ba_100000/"
seed_df = pd.read_csv(seed_dirs+sw[k][0])
temp = seed_df[['E','S']].shift([0,1])
# Inc_t = (E_t-1 - E_t) - (S_t - S_t-1)
seed_df['incidence'] = (temp['E_1'] - temp['E_0']) - \
                    (temp['S_0'] - temp['S_1'])
seed_df = seed_df[(seed_df['E'] > 0)|(seed_df['I'] > 0)
                         ].fillna(0)
seed_df.replace([np.inf, -np.inf], 0, inplace=True)
    
scaler = joblib.load(f'ba100k_lstm_4_001_s10.pkl')
model = predict_Beta_I.load_model(f'ba100k_lstm_4_001_s10.keras')
window_size=4
predicted_days = [4,5]

In [ ]:
seed_df[['Beta']].shift(np.arange(window_size)
                            ).iloc[4]

In [ ]:
inp = seed_df[['Beta']].shift(np.arange(window_size)
                            ).iloc[predicted_days[0]
                                  ].values
#[t, t-1, t-2, t-3]
inp = np.log(inp+1e-7)
#[t-3, ... t]
sc_inp = scaler.transform(inp[::-1].reshape(-1, 1))


In [ ]:
sc_inp.shape

In [ ]:
np.array(sc_inp)

In [ ]:
%%timeit
sc = sc_inp.reshape(1, 4, 1)
sc = tf.convert_to_tensor(sc)
model.predict(sc, verbose=0, batch_size=len(sc))

In [ ]:
%%timeit
sc = sc_inp.reshape(1, 4, 1)
sc = tf.convert_to_tensor(sc)
model(sc)

In [ ]:
%%timeit
sc = sc_inp.reshape(1, 4, 1)
sc = tf.convert_to_tensor(sc)
model.predict_on_batch(sc)

In [ ]:
model.summary()

In [ ]:
%%timeit
sc = sc_inp.reshape(1, 4, 1)
for i in range(predicted_days[0], 
               30): 
    pred = model(sc)
    
    # add pred in the beginning: [y_hat, t, t-1, t-2]
    result = np.empty_like(sc)
    result[:,:1] = pred
    result[:,1:] = sc[:,:-1]
    
    sc = result

In [ ]:
preds = []
sc = sc_inp.reshape(1, 4, 1)
for i in range(predicted_days[0], 
               30): 
    pred = model(sc)
    
    # add pred in the beginning: [y_hat, t, t-1, t-2]
    result = np.empty_like(sc)
    result[:,:1] = pred
    result[:,1:] = sc[:,:-1]
    preds.append(pred)
    sc = result

In [ ]:
pred

In [ ]:
scaler.transform(np.array([np.log(1e-7)]).reshape(-1, 1))

In [ ]:
tf.convert_to_tensor([[0]])

In [ ]:
preds2 = []
sc = sc_inp.reshape(1, 4, 1)
for i in range(predicted_days[0], 
               30): 
    pred = model.predict(sc, verbose=0)
    
    # add pred in the beginning: [y_hat, t, t-1, t-2]
    result = np.empty_like(sc)
    result[:,:1] = pred
    result[:,1:] = sc[:,:-1]
    preds2.append(pred)
    sc = result

In [ ]:
fastp = tf.convert_to_tensor(preds).numpy()
fastp = scaler.inverse_transform(fastp[::,0])
fastp = np.exp(fastp.flatten())
plt.plot(fastp)

sp = np.array(preds2)
sp = scaler.inverse_transform(sp[::,0])
sp = np.exp(sp.flatten())
plt.plot(sp)

k = seed_df.Beta.iloc[4:31].values
#k = np.log(k+1e-7)
#[t-3, ... t]
#k = scaler.transform(k[::-1].reshape(-1, 1))
plt.plot(k)

In [ ]:
%%timeit
sc = sc_inp.reshape(1, 4, 1)
for i in range(predicted_days[0], 
               30): 
    pred = model.predict_on_batch(sc)
    
    # add pred in the beginning: [y_hat, t, t-1, t-2]
    result = np.empty_like(sc)
    result[:,:1] = pred
    result[:,1:] = sc[:,:-1]
    
    sc = result

In [ ]:
%%timeit
scaler.inverse_transform(tf.convert_to_tensor(predicted_beta)[::,0])

In [ ]:
%%timeit
scaler.inverse_transform(tf.convert_to_tensor(predicted_beta
                                             ).numpy()[::,0])

In [ ]:
tf.convert_to_tensor(predicted_beta
                                             ).numpy()[::,0]

In [ ]:
r = scaler.inverse_transform(tf.convert_to_tensor(predicted_beta
                                             ).numpy()[::,0])
np.exp(r.flatten())

In [ ]:
tf.__version__

In [ ]:
model(sc)

In [ ]:
model.predict(sc, verbose=0)

In [ ]:
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, perc_switch=0.005,
             suff_m='ba100k', suff='_inc_05')

In [ ]:
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, perc_switch=0.0075,
             suff_m='ba100k', suff='_inc_075')

In [ ]:
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,
             suff_m='ba100k', suff='_inc_01')

In [ ]:
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, perc_switch=0.0125,
             suff_m='ba100k', suff='_inc_0125')

In [ ]:
apply_methods(seed_dirs="../new_ba_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, perc_switch=0.015,
             suff_m='ba100k', suff='_inc_015')

## sw

In [ ]:
sw = pd.read_csv('test_files.csv').values
k = 22000
apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[k:k+1], on_incidence=True,
              idx_s=0, idx_e=5, show_fig_flag=True,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,
             suff_m='sw100k')


In [ ]:
sw = pd.read_csv('test_files.csv').values
apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.0005,
             suff_m='sw100k', suff='_inc_05')

In [ ]:
sw = pd.read_csv('test_files.csv').values
apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.00075,
             suff_m='sw100k', suff='_inc_075')

In [ ]:
sw = pd.read_csv('test_files.csv').values
apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.01,
             suff_m='sw100k', suff='_inc_1')

In [ ]:
sw = pd.read_csv('test_files.csv').values
apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.0125,
             suff_m='sw100k', suff='_inc_125')

In [ ]:
sw = pd.read_csv('test_files.csv').values
apply_methods(seed_dirs="../new_sw_100000/",
              seed_numbers=sw[::10], on_incidence=True,
              idx_s=0, idx_e=4, show_fig_flag=False,
             is_filename=True, sigma=0.3, gamma=0.2, 
              perc_switch=0.015,
             suff_m='sw100k', suff='_inc_15')